# Taller 8 · MEDIQA-CORR 2024: implementación del pipeline

Grupo: David Moreno y Dierick Brochero · PLN, Javeriana

Instancia el pipeline diseñado en el Taller 7 sobre el dataset MS y resuelve las tres subtareas: **A** (¿hay error?), **B** (¿en qué oración?) y **C** (oración corregida). Las funciones viven en `src/mediqa_corr/`, un módulo por caja; este notebook las llama en orden.

| Caja | Módulo | Modelo |
|---|---|---|
| 1. Datos | `data.py` | pandas, TF-IDF |
| 2. Detector (A, B) | `detector.py` | `emilyalsentzer/Bio_ClinicalBERT` |
| 3. Corrector (C) | `corrector.py` | `google/flan-t5-base` |
| 4. Evaluación | `evaluation.py`, `baselines.py` | `rouge`, `bert_score` (deberta-xlarge-mnli) |

En Colab: Entorno de ejecución → Cambiar tipo de entorno → GPU. Con `QUICK_TEST = True` corre en pocos minutos para comprobar que todo funciona. La corrida completa ya ejecutada está en `notebooks/ejecucion_rtx3090.ipynb`.

In [ ]:
# Instalación (Colab). En local basta con: pip install -r requirements.txt
import os, sys
if "google.colab" in sys.modules and not os.path.exists("NLP_JAVE_TALLER08_202601"):
    !git clone -q https://github.com/davidm094/NLP_JAVE_TALLER08_202601.git
    %cd NLP_JAVE_TALLER08_202601
    !pip install -q -r requirements.txt
if not os.path.exists("src"):      # abierto desde notebooks/
    os.chdir("..")
sys.path.insert(0, "src")

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support
from mediqa_corr import data, detector, corrector, evaluation, baselines, pipeline
from mediqa_corr.config import TrainConfig

QUICK_TEST = False            # True: ~200 textos y 1 época por modelo
DATA_PATH = "data/MEDIQA-CORR-2024-MS-TrainingData.csv"
cfg = TrainConfig()
if QUICK_TEST:
    cfg.det_epochs, cfg.cor_epochs = 1, 1
pipeline.set_seed(cfg.seed)

## Caja 1 · Datos: segmentación oficial, casos gemelos y división por grupo

In [ ]:
df = data.load_data(DATA_PATH)
df["group"] = data.make_case_groups(df)          # versiones correcta y con error del mismo caso
stats = data.describe_dataset(df)
if QUICK_TEST:
    df = data.sample_groups(df, 110)
train_df, dev_df, test_df = data.split_train_dev_test(df)
for a, b in [(train_df, dev_df), (train_df, test_df), (dev_df, test_df)]:
    assert set(a["group"]).isdisjoint(set(b["group"]))   # ningún caso compartido
train_sent, dev_sent = data.build_sentence_table(train_df), data.build_sentence_table(dev_df)
print(len(train_df), len(dev_df), len(test_df), "textos ->", len(train_sent), "oraciones de train")

## Pisos en test (lo que se saca sin modelo)

In [ ]:
print("A, siempre error:", baselines.baseline_always_error(test_df))
print("B, nunca error:  ", baselines.baseline_never_error(test_df))
print("Última oración:  ", baselines.baseline_last_sentence(test_df))
print("C, copiar:       ", baselines.baseline_copy_sentence(test_df))
evaluation.release_bert_scorer()     # libera ~3 GB antes de entrenar

## Caja 2 · Detector (subtareas A y B)

In [ ]:
det_trainer, det_tok = detector.train_error_classifier(
    train_sent, dev_sent, model_name=cfg.detector_name, epochs=cfg.det_epochs, lr=cfg.det_lr,
    batch_size=cfg.det_batch_size)
det_model = det_trainer.model

In [ ]:
def scores(d):
    return [detector.predict_sentence_scores(data.parse_sentences(r["Sentences"]), data.clean(r["Text"]), det_model, det_tok)
            for _, r in d.iterrows()]

threshold = detector.choose_threshold(scores(dev_df), dev_df["Error Flag"])   # umbral elegido en dev
test_scores = scores(test_df)
flags = [detector.decide_flag(s, threshold) for s in test_scores]
ids = [detector.pick_error_sentence(s, f) for s, f in zip(test_scores, flags)]
p, r, f1, _ = precision_recall_fscore_support(test_df["Error Flag"], flags, average="binary", zero_division=0)
print("Umbral:", threshold)
print("A:", round(evaluation.eval_flag_accuracy(test_df["Error Flag"], flags), 4),
      "| B:", round(evaluation.eval_sentence_accuracy(test_df["Error Sentence ID"], ids), 4),
      f"| precisión {p:.3f}, recall {r:.3f}")

## Caja 3 · Corrector (subtarea C)

In [ ]:
cor_trainer, cor_tok = corrector.train_corrector(
    corrector.build_correction_pairs(train_df), corrector.build_correction_pairs(dev_df),
    model_name=cfg.corrector_name, epochs=cfg.cor_epochs, lr=cfg.cor_lr, batch_size=cfg.cor_batch_size)
cor_model = cor_trainer.model

ver = test_df[test_df["Error Flag"] == 1]       # corrector con la oración errónea real (cota superior de C)
preds = [corrector.generate_correction(corrector.build_correction_source(s, t), cor_model, cor_tok, fallback=s)
         for s, t in zip(ver["Error Sentence"], ver["Text"])]
res_c = evaluation.eval_correction(preds, [data.clean(s) for s in ver["Corrected Sentence"]]); res_c.pop("per_text")
print("Corrector con la oración real:", res_c)

## Caja 4 · Pipeline completo y evaluación en test

In [ ]:
pred_test = pipeline.run_pipeline(test_df, det_model, det_tok, cor_model, cor_tok, threshold)
resultados = evaluation.evaluate_all(pred_test, test_df)
print(resultados)
pipeline.write_predictions(pred_test, "results/predicciones_test.csv")   # también escribe el .txt oficial